In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("bronze_table_name", "", "Bronze Table Name")
dbutils.widgets.text("target_table_name", "", "Target Table Name")
dbutils.widgets.text("load_type", "", "Load Type")
dbutils.widgets.text("merge_key", "", "Merge Key")
dbutils.widgets.text("scd_type", "", "SCD Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
bronze_table_name = dbutils.widgets.get("bronze_table_name")
target_table_name = dbutils.widgets.get("target_table_name")
load_type = dbutils.widgets.get("load_type")
merge_key = dbutils.widgets.get("merge_key")
scd_type = dbutils.widgets.get("scd_type")

In [0]:
bronze_df_original_count = spark.read.table(f"amperedrive_giga.bronze.{bronze_table_name}").count()

In [0]:
bronze_filtered_df = spark.read.table(f"amperedrive_giga.bronze.{bronze_table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")


In [0]:
from pyspark.sql.functions import current_timestamp

updated_bronze_df = bronze_filtered_df.withColumn("insert_timestamp", current_timestamp()).withColumn("updated_at", current_timestamp())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

target_fqn = f"amperedrive_giga.silver.{target_table_name}"
load_type_norm = load_type.strip().upper()
scd_type_norm = scd_type.strip()
merge_keys = [k.strip() for k in merge_key.split(",") if k.strip()]  # supports composite keys: "col1,col2"

# Columns excluded from change detection (audit / technical columns)
AUDIT_COLS = {"insert_timestamp", "updated_at", "landing_timestamp"}
SCD2_COLS = ["effective_start_date", "effective_end_date", "is_current"]

rows_inserted, rows_updated = 0, 0


def last_merge_metrics():
    m = DeltaTable.forName(spark, target_fqn).history(1).select("operationMetrics").first()[0]
    return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))


if load_type_norm == "FULL_OVERWRITE":
    updated_bronze_df.write.mode("overwrite").saveAsTable(target_fqn)
    rows_loaded = updated_bronze_df.count()
    rows_inserted, rows_updated = rows_loaded, rows_loaded

elif load_type_norm == "INCREMENTAL_APPEND":
    updated_bronze_df.write.mode("append").saveAsTable(target_fqn)
    rows_loaded = updated_bronze_df.count()
    rows_inserted = rows_loaded

elif load_type_norm == "INCREMENTAL_MERGE":
    if not merge_keys:
        raise ValueError("merge_key is required for INCREMENTAL_MERGE")
    if scd_type_norm not in ("1", "2"):
        raise ValueError(f"Unsupported scd_type '{scd_type}' for INCREMENTAL_MERGE. Expected 1 or 2.")

    # Keep only the latest record per merge key within this landing batch, then stamp audit columns
    w = Window.partitionBy(*merge_keys).orderBy(F.col("updated_at").desc_nulls_last())
    src_df = (bronze_filtered_df
              .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
              .withColumn("insert_timestamp", F.current_timestamp())
              .withColumn("updated_at", F.current_timestamp()))
    rows_loaded = src_df.count()

    key_cond = " AND ".join(f"t.`{k}` = s.`{k}`" for k in merge_keys)
    compare_cols = [c for c in src_df.columns if c not in merge_keys and c not in AUDIT_COLS]
    change_cond = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in compare_cols) or "false"

    if scd_type_norm == "1":
        if not spark.catalog.tableExists(target_fqn):
            src_df.write.saveAsTable(target_fqn)
            rows_inserted = rows_loaded
        else:
            # SCD1: overwrite changed attributes in place (keep original insert_timestamp), insert new keys
            update_set = {f"`{c}`": f"s.`{c}`" for c in src_df.columns if c != "insert_timestamp"}
            (DeltaTable.forName(spark, target_fqn).alias("t")
                .merge(src_df.alias("s"), key_cond)
                .whenMatchedUpdate(condition=change_cond, set=update_set)
                .whenNotMatchedInsertAll()
                .execute())
            rows_inserted, rows_updated = last_merge_metrics()

    else:  # SCD Type 2
        if not spark.catalog.tableExists(target_fqn):
            (src_df.withColumn("effective_start_date", F.current_timestamp())
                   .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
                   .withColumn("is_current", F.lit(True))
                   .write.saveAsTable(target_fqn))
            rows_inserted = rows_loaded
        else:
            missing = [c for c in SCD2_COLS if c not in spark.table(target_fqn).columns]
            if missing:
                raise ValueError(f"Target {target_fqn} is missing SCD2 columns {missing}. "
                                 "Use a new target table or add these columns first.")
            tgt = DeltaTable.forName(spark, target_fqn)
            current_df = tgt.toDF().filter("is_current = true")

            # Source rows whose attributes changed vs the current version -> need a new version row
            changed_df = (src_df.alias("s")
                          .join(current_df.alias("t"), on=F.expr(key_cond), how="inner")
                          .filter(F.expr(change_cond))
                          .select("s.*"))

            # Staged source: NULL merge keys force INSERT of new versions; real keys drive expiry / new-key inserts
            staged_df = (changed_df.select(*[F.lit(None).cast(src_df.schema[k].dataType).alias(f"__mk_{k}") for k in merge_keys], "*")
                         .unionByName(src_df.select(*[F.col(k).alias(f"__mk_{k}") for k in merge_keys], "*")))

            staged_cond = " AND ".join(f"t.`{k}` = s.`__mk_{k}`" for k in merge_keys) + " AND t.is_current = true"
            insert_values = {f"`{c}`": f"s.`{c}`" for c in src_df.columns}
            insert_values.update({
                "effective_start_date": "current_timestamp()",
                "effective_end_date": "CAST(NULL AS TIMESTAMP)",
                "is_current": "true",
            })
            (tgt.alias("t")
                .merge(staged_df.alias("s"), staged_cond)
                .whenMatchedUpdate(condition=change_cond, set={
                    "is_current": "false",
                    "effective_end_date": "current_timestamp()",
                    "updated_at": "current_timestamp()",
                })
                .whenNotMatchedInsert(values=insert_values)
                .execute())
            rows_inserted, rows_updated = last_merge_metrics()

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL_OVERWRITE, INCREMENTAL_MERGE or INCREMENTAL_APPEND.")

print(f"{load_type_norm} (scd_type={scd_type_norm or 'n/a'}) -> {target_fqn}: loaded={rows_loaded}, inserted={rows_inserted}, updated={rows_updated}")

In [0]:
# rows_loaded / rows_inserted / rows_updated are set by the load cell above for each load_type
result = {
    "rows_extracted": str(bronze_df_original_count),
    "rows_loaded": str(rows_loaded),
    "rows_inserted": str(rows_inserted),
    "rows_updated": str(rows_updated)
}

dbutils.notebook.exit(result)